# 02 — Create 1 km × 1 km Grid for West Java

This notebook is part of the reproducible workflow for the multi-source geospatial AI framework.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
from pathlib import Path

ROOT = Path('..')
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

boundary_path = RAW / 'boundary_kabupaten_kota_jawa_barat_27_big.geojson'
boundary = gpd.read_file(boundary_path).to_crs('EPSG:32748')
province = boundary.dissolve()

minx, miny, maxx, maxy = province.total_bounds
cell_size = 1000
xs = np.arange(minx, maxx + cell_size, cell_size)
ys = np.arange(miny, maxy + cell_size, cell_size)

polys = []
ids = []
for i, x in enumerate(xs[:-1]):
    for j, y in enumerate(ys[:-1]):
        geom = box(x, y, x + cell_size, y + cell_size)
        if geom.intersects(province.geometry.iloc[0]):
            ids.append(f'G_{i:04d}_{j:04d}')
            polys.append(geom)

grid = gpd.GeoDataFrame({'grid_id': ids}, geometry=polys, crs='EPSG:32748')
grid['area_km2'] = grid.geometry.area / 1e6
# Assign district by centroid spatial join
centroids = grid.copy()
centroids.geometry = centroids.geometry.centroid
joined = gpd.sjoin(centroids, boundary, how='left', predicate='within')
name_col = 'namobj' if 'namobj' in boundary.columns else boundary.columns[0]
grid['district'] = joined[name_col].values
# Output
grid.to_file(PROCESSED / 'grid_1km_jawa_barat.gpkg', layer='grid_1km', driver='GPKG')
grid.to_crs('EPSG:4326').to_file(PROCESSED / 'grid_1km_jawa_barat.geojson', driver='GeoJSON')
print(grid.shape)
